# Porto Seguro Safe Driver Prediction DSPP

## Hypothesis:
### Including vehicle-related `car_` features improves prediction of claims
model's ability to correctly identify actual claims (recall), without a disproportionate drop in precision, relative to the baseline model which uses only individual and regional `ind_`/`reg_` features.

This notebook builds and compares baseline vs comparison feature sets across two model
types (logistic regression and LightGBM), first on a stratified 15% sample, then on the
full ~595k-row dataset, to test whether the finding holds at different sample sizes.

Dataset: (https://www.kaggle.com/datasets/pipergragg/porto-seguros-safe-driver-prediction-dataset/data)

In [1]:
import pandas as pd
df = pd.read_csv('/content/train.csv', on_bad_lines='warn')
print(df.shape)
#checking the shape of the datset to make sure the correct one was downloaded and read fine by pandas

(595212, 59)


In [2]:
print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True))
#seeing how frequent the target is to decide if random sampling would be safe

target
0    573518
1     21694
Name: count, dtype: int64
target
0    0.963552
1    0.036448
Name: proportion, dtype: float64


In [3]:
#because the target is about 3.6% , stratified sampling makes more sense to preserve that ratio
# stratified_sample = df.groupby('target', group_keys=False).apply(lambda x: x.sample(frac=0.15, random_state=1))
stratified_sample = df
# decided not to stratify sample and instead use the full dataset out of curiousity and seeing variance in hypothesis conclusions
print(stratified_sample.shape)
print(stratified_sample['target'].value_counts(normalize='true'))
#checking that the stratfied sample has kept the 3.6% target

(595212, 59)
target
0    0.963552
1    0.036448
Name: proportion, dtype: float64


In [4]:
# Getting an idea for where missing values turn up
(stratified_sample == -1).sum().sort_values(ascending=False).head(25)

,0
ps_car_03_cat,411231
ps_car_05_cat,266551
ps_reg_03,107772
ps_car_14,42620
ps_car_07_cat,11489
ps_ind_05_cat,5809
ps_car_09_cat,569
ps_ind_02_cat,216
ps_car_01_cat,107
ps_ind_04_cat,83


In [5]:
# removing columns with missing
missing_counts = (stratified_sample == -1).sum()
cols_with_missing = missing_counts[missing_counts > 0].index.tolist()
stratified_sample = stratified_sample.drop(columns=cols_with_missing)
print(cols_with_missing)

['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_reg_03', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_05_cat', 'ps_car_07_cat', 'ps_car_09_cat', 'ps_car_11', 'ps_car_12', 'ps_car_14']


In [6]:
# getting an idea of the naming convention for the columnms
cols = stratified_sample.columns
print(cols)

Index(['id', 'target', 'ps_ind_01', 'ps_ind_03', 'ps_ind_06_bin',
       'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin',
       'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14',
       'ps_ind_15', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin',
       'ps_reg_01', 'ps_reg_02', 'ps_car_04_cat', 'ps_car_06_cat',
       'ps_car_08_cat', 'ps_car_10_cat', 'ps_car_11_cat', 'ps_car_13',
       'ps_car_15', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_calc_04',
       'ps_calc_05', 'ps_calc_06', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09',
       'ps_calc_10', 'ps_calc_11', 'ps_calc_12', 'ps_calc_13', 'ps_calc_14',
       'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_calc_18_bin',
       'ps_calc_19_bin', 'ps_calc_20_bin'],
      dtype='object')


In [7]:
#seen that there is 4 prefixes that cover all the columns
for prefix in ['ind', 'reg', 'car', 'calc']:
    print(prefix, [c for c in cols if prefix in c])

ind ['ps_ind_01', 'ps_ind_03', 'ps_ind_06_bin', 'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_ind_15', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin']
reg ['ps_reg_01', 'ps_reg_02']
car ['ps_car_04_cat', 'ps_car_06_cat', 'ps_car_08_cat', 'ps_car_10_cat', 'ps_car_11_cat', 'ps_car_13', 'ps_car_15']
calc ['ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_calc_04', 'ps_calc_05', 'ps_calc_06', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09', 'ps_calc_10', 'ps_calc_11', 'ps_calc_12', 'ps_calc_13', 'ps_calc_14', 'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_calc_18_bin', 'ps_calc_19_bin', 'ps_calc_20_bin']


In [8]:
#checking if all of the features with "_cat" are actually categorical feature by checking the count of their unique values which is by definition also the cardinality
stratified_sample.filter(like='_cat').nunique().sort_values(ascending=False)

,0
ps_car_11_cat,104
ps_car_06_cat,18
ps_car_04_cat,10
ps_car_10_cat,3
ps_car_08_cat,2


In [9]:
# Removing ps_car_11_cat because it has 104 categories whichn isnt realistically usable in a logistic regression model
stratified_sample = stratified_sample.drop(columns=['ps_car_11_cat'])

In [10]:
#splitting column names into categories in anticipation of different processing requirements of the columns for modelling down the line
binaryCols = [c for c in cols if 'bin' in c]
categoricalCols = [c for c in cols if 'cat' in c]
numericalCols = [c for c in cols if c not in binaryCols + categoricalCols and c not in ['id', 'target']]
print('binary:', len(binaryCols), 'categorical:', len(categoricalCols), 'numerical:', len(numericalCols))

binary: 17 categorical: 5 numerical: 22


In [11]:
# processing the categorical columns into binary flags
cat_cols_remaining = [c for c in stratified_sample.columns if 'cat' in c]
stratified_sample = pd.get_dummies(stratified_sample, columns=cat_cols_remaining, drop_first=True)
# added drop_first = True to prevent creating categories that are duplicates

In [12]:
#Scaling the remaining columns that arent categorical or Binary
#made sure it was safe to scale by removing missing value containing columns earlier
from sklearn.preprocessing import StandardScaler

exclude = ['id', 'target'] + binaryCols
# dtype != bool makes sure that we dont scale the binary flags
num_cols_all = [c for c in stratified_sample.columns if c not in exclude and stratified_sample[c].dtype != 'bool']

scaler = StandardScaler()
stratified_sample[num_cols_all] = scaler.fit_transform(stratified_sample[num_cols_all])

In [13]:
#checking healthyness of the data now
print(stratified_sample.shape)
print(stratified_sample.isna().sum().sum())  # should be 0
stratified_sample.head()
# shape shows that row count is healthy
# isna shows no missing cells are remaining
# head shows scaling worked properly

(595212, 70)
0


,id,target,ps_ind_01,ps_ind_03,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,...,ps_car_06_cat_11,ps_car_06_cat_12,ps_car_06_cat_13,ps_car_06_cat_14,ps_car_06_cat_15,ps_car_06_cat_16,ps_car_06_cat_17,ps_car_08_cat_1,ps_car_10_cat_1,ps_car_10_cat_2
0,7,0,0.050218,0.213594,0,1,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,9,0,-0.453868,0.954362,0,0,1,0,0,0,...,True,False,False,False,False,False,False,True,True,False
2,13,0,1.562477,1.695130,0,0,1,0,0,0,...,False,False,False,True,False,False,False,True,True,False
3,16,0,-0.957955,-0.897559,1,0,0,0,0,0,...,True,False,False,False,False,False,False,True,True,False
4,17,0,-0.957955,-1.638327,1,0,0,0,0,0,...,False,False,False,True,False,False,False,True,True,False


In [14]:
#creating features for the baseline model which only includes individual and regional related columns, so nothing related to the car. Also has none of the calc factors either.
baseline_features = [c for c in stratified_sample.columns if ('ind' in c or 'reg' in c) and c not in ['id', 'target']]
#comparision features adds just the car related columns to the feature set
comparison_features = baseline_features + [c for c in stratified_sample.columns if 'car' in c]
print('baseline feature count:', len(baseline_features))
print('comparison feature count:', len(comparison_features))
# the seemingly high difference in feature count is just because the car colums are categorical with several categories. becase we turned them into binary flag columns , each category now counts as a column.

baseline feature count: 17
comparison feature count: 48


In [15]:
from sklearn.model_selection import train_test_split

X_base = stratified_sample[baseline_features]
X_comp = stratified_sample[comparison_features]
y = stratified_sample['target']
#used stratify = y for the same reasons as the initial dataset (when I was sampling that) , which is to maintain the 3.6% frequency of the target
X_base_train, X_base_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, stratify=y, random_state=42)
#made sure the random state is the same so the comparision is genuinely apples to apples as the rows will be the same, only the features will differ
X_comp_train, X_comp_test, _, _ = train_test_split(X_comp, y, test_size=0.2, stratify=y, random_state=42)

print(X_base_train.shape, X_base_test.shape)
print(X_comp_train.shape, X_comp_test.shape)

(476169, 17) (119043, 17)
(476169, 48) (119043, 48)


In [16]:
from sklearn.linear_model import LogisticRegression

#using balanced class weight so there is more weight on the claim class which we already know is infrequent, since we are building a model to predict that.
# increased the max iter because models failed to converge on 100
model_baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_baseline.fit(X_base_train, y_train)

model_comparison = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_comparison.fit(X_comp_train, y_train)

print("converged")

converged


In [17]:
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report

for name, model, X_test in [('Baseline factors', model_baseline, X_base_test), ('Comparison (baseline + car)', model_comparison, X_comp_test)]:
    #.predict_proba shows predicted probability for claim which is needed to calculate PR AUC
    probs = model.predict_proba(X_test)[:, 1]
    # using PR AUC over ROC AUC because the target is so infrequent at 3.6% , the latter metric might be misleading when it comes to comparing models if one model just predicted no claims for everyone
    pr_auc = average_precision_score(y_test, probs)
    #using 0.5 as the cutoff point for the probability to be considered as a prediction for building the confusion matrix
    preds = (probs >= 0.5).astype(int)
    print(f'{name}')
    print('PR AUC:', round(pr_auc, 4))
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, digits=3))
    print()

Baseline factors
PR AUC: 0.0536
[[69135 45569]
 [ 2009  2330]]
              precision    recall  f1-score   support

           0      0.972     0.603     0.744    114704
           1      0.049     0.537     0.089      4339

    accuracy                          0.600    119043
   macro avg      0.510     0.570     0.417    119043
weighted avg      0.938     0.600     0.720    119043


Comparison (baseline + car)
PR AUC: 0.0561
[[70617 44087]
 [ 2003  2336]]
              precision    recall  f1-score   support

           0      0.972     0.616     0.754    114704
           1      0.050     0.538     0.092      4339

    accuracy                          0.613    119043
   macro avg      0.511     0.577     0.423    119043
weighted avg      0.939     0.613     0.730    119043




In [18]:
# using a LGM to see if its ability to split on high cardinality factors breaks a limitation of the GLMs , which will tell us if the results seen on the GLM comparision are due to the factors we are testing , or the nature of GLMs
import lightgbm as lgb
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report

#Baseline LGBM
#using balanced class weight so there is more weight on the claim class which we already know is infrequent, since we are building a model to predict that.
# keeping the random state the same as the GLMs so we compare on the same rows
lgb_baseline = lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=200, verbose=-1)
lgb_baseline.fit(X_base_train, y_train)
probs_base = lgb_baseline.predict_proba(X_base_test)[:, 1]
preds_base = (probs_base >= 0.5).astype(int)

print('LightGBM (baseline factors)')
print('PR AUC:', round(average_precision_score(y_test, probs_base), 4))
print(confusion_matrix(y_test, preds_base))
print(classification_report(y_test, preds_base, digits=3))
print()

# Comparison LGBM
lgb_comparison = lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=200, verbose=-1)
lgb_comparison.fit(X_comp_train, y_train)
probs_comp = lgb_comparison.predict_proba(X_comp_test)[:, 1]
preds_comp = (probs_comp >= 0.5).astype(int)

print('LightGBM (comparison factors)')
print('PR AUC:', round(average_precision_score(y_test, probs_comp), 4))
print(confusion_matrix(y_test, preds_comp))
print(classification_report(y_test, preds_comp, digits=3))

LightGBM (baseline factors)
PR AUC: 0.0577
[[72548 42156]
 [ 2111  2228]]
              precision    recall  f1-score   support

           0      0.972     0.632     0.766    114704
           1      0.050     0.513     0.091      4339

    accuracy                          0.628    119043
   macro avg      0.511     0.573     0.429    119043
weighted avg      0.938     0.628     0.742    119043


LightGBM (comparison factors)
PR AUC: 0.059
[[75280 39424]
 [ 2106  2233]]
              precision    recall  f1-score   support

           0      0.973     0.656     0.784    114704
           1      0.054     0.515     0.097      4339

    accuracy                          0.651    119043
   macro avg      0.513     0.585     0.440    119043
weighted avg      0.939     0.651     0.759    119043

